# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

De façon similaire au TP sur les maillages, il sera demandé de pouvoir retrouver une tangente et un rayon en chaque point, ainsi qu’un point sur la surface définie par le nuage de points.

In [156]:
# Imports
import numpy as np
import plotly.graph_objects as go
import open3d as o3d

from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 200                    # Nombre de points
NUM_NEIGHBORS = 5                   # Nombre de voisin par point
k = 2                               # Pour le calcul de k_near_neighbors

# Type de génération
GENERATION_TYPE = 'surface'

### Générer le nuage de points

In [157]:

def generate_points(size, num_points, distribution):
    """
    Générer des points en 3D en fonction de la taille spécifiée, du nombre de points et de la distribution.
    Paramètres :
    - size (float) : La taille de l'espace dans lequel les points seront générés.
    - num_points (int) : Le nombre de points à générer.
    - distribution (str) : Le type de distribution des points.
    Renvoie :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D générés.
    """
    if distribution == 'surface':
        edge = int(np.sqrt(num_points))
        NUM_POINTS = edge**2
        x = np.linspace(-size, size, edge)
        y = np.linspace(-size, size, edge)
        x, y = np.meshgrid(x, y)
        z = x**2 - y**2
        z *= size/np.max(z)
        points = np.column_stack((x.flatten(), y.flatten(), z.flatten()))
        center = [0, 0, -size]

    elif distribution == 'regular_phere':
        indices = np.arange(0, num_points, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices/num_points)
        theta = np.pi * (1 + 5**0.5) * indices
        x = size * np.cos(theta) * np.sin(phi)
        y = size * np.sin(theta) * np.sin(phi)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))
        center = [0, 0, 0]

    elif distribution == 'random_phere':
        theta = 2 * np.pi * np.random.rand(num_points)
        phi = np.arccos(2 * np.random.rand(num_points) - 1)
        x = size * np.sin(phi) * np.cos(theta)
        y = size * np.sin(phi) * np.sin(theta)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))
        center = [0, 0, 0]

    else:
        fichier_xyz = "models3D/" + distribution + ".xyz"
        points = np.loadtxt(fichier_xyz)
        echant = np.linspace(0, points.shape[0] - 1, NUM_POINTS, dtype=int)
        points = points[echant]
        points[:, 1] = -points[:, 1]
        points *= size/np.max(points)
        centre = np.mean(points)

    return points, center


points, center = generate_points(size=5, num_points=NUM_POINTS, distribution=GENERATION_TYPE)

In [158]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

fig.show()

### Obtenir les voisins

In [159]:
def get_direct_neighbors(points, num_neighbors):
    """
    Trouver les voisins directs de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - num_neighbors (int) : Le nombre de voisins à trouver.
    Renvoie :
    - indices (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    """
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='kd_tree').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = get_direct_neighbors(points=points, num_neighbors=NUM_NEIGHBORS)

In [160]:
fig = new_fig()

# Ajouter les voisins directs à la figure
for i_point, point in enumerate(points):
    for i_neighbor in direct_neighbors[i_point]:
        neighbor = points[i_neighbor]
        add_line(point, neighbor, fig, 'blue')
        
fig.show()

### Obtenir les k plus proches voisins

In [162]:
def get_k_near_neighbors(direct_neighbors, k):
    """
    Trouver les k-voisins de chaque point.
    Paramètres :
    - direct_neighbors (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    - k (int) : La profondeur de la recherche.
    Renvoie :
    - k_near_neighbors (list) : Une liste de longueur num_points contenant les indices des k-voisins de chaque point.
    """
    k_near_neighbors = [set() for _ in range(NUM_POINTS)]
    
    for i_point in range(NUM_POINTS):
        k_near_neighbors[i_point].add(i_point)

    for _ in range(k):
        for i_point in range(NUM_POINTS):
            for i_neighbor in direct_neighbors[i_point]:
                k_near_neighbors[i_point].update(direct_neighbors[i_neighbor])
    
    for i_point in range(NUM_POINTS):
        k_near_neighbors[i_point].remove(i_point)
    
    return k_near_neighbors


k_near_neighbors = get_k_near_neighbors(direct_neighbors=direct_neighbors, k=k)

IndexError: index 196 is out of bounds for axis 0 with size 196

In [ ]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

# Ajouter un point d'example à la figure
example = np.random.randint(0, NUM_POINTS)
add_point(points[example], fig, 'red')

# Ajouter les k-voisins de l'exemple à la figure
for neighbor in k_near_neighbors[example]:
    add_point(points[neighbor], fig, 'orange')

to_plot_neighbors = [example]
for _ in range(k):
    new_neighbors = []
    for point in to_plot_neighbors:
        for neighbor in direct_neighbors[point]:
            new_neighbors.append(neighbor)
            add_line(points[point], points[neighbor], fig, 'blue')
    to_plot_neighbors = new_neighbors


fig.show()

### Calculer les normales sortantes

In [ ]:
def get_normal(points, neighbors):
    """
    Trouver les vecteurs normaux de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - neighbors (list) : Une liste de longueur num_points contenant les indices des k-voisins de chaque point.
    Renvoie :
    - normals (list) : Une liste de longueur num_points contenant les vecteurs normaux de chaque point.
    """
    T1 = []
    T2 = []
    normals = []
    
    for p in range(NUM_POINTS):
        neighbor_points = points[list(neighbors[p])]
        
        # Faire l'ACP
        pca = PCA(n_components=3)
        pca.fit(neighbor_points)
        
        T1.append(pca.components_[0])
        T2.append(pca.components_[1])

        normal_vector = pca.components_[2]
        if np.linalg.norm(center - points[p] + normal_vector) > np.linalg.norm(center - points[p]) :
            normal_vector =  - normal_vector
        normals.append(normal_vector)
    
    return normals, T1, T2


normals, T1, T2 = get_normal(points=points, neighbors=k_near_neighbors)

In [ ]:
fig = new_fig()

# Ajouter les points à la figure
add_points(points, fig, 'magenta')

# Ajouter les points à la figure
add_point(center, fig, 'cyan')

# Ajouter les vecteurs normaux à la figure
for point, normal in zip(points, normals):
    add_line(point, point + normal, fig, 'orange')

fig.show()

### Trouver les rayons

In [ ]:
def get_rayons(points, direct_neighbors):
    """
    Trouver les rayons de chaque point.
    Paramètres :
    - points (ndarray) : Un tableau de forme (num_points, 3) contenant les points en 3D.
    - direct_neighbors (ndarray) : Un tableau de forme (num_points, num_neighbors) contenant les indices des voisins directs.
    Renvoie :
    - rayons (list) : Une liste de longueur num_points contenant les rayons de chaque point.
    """
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points=points, direct_neighbors=direct_neighbors)

In [ ]:
fig = new_fig()

# Ajouter les points à la figure
for point, rayon, T1_point, T2_point in zip(points, rayons, T1, T2):
    add_circle(point, rayon, T1_point, T2_point, fig, 'lightgreen')

fig.show()

### Estimer la surface

In [ ]:
# Créer un nuage de points
point_cloud = o3d.geometry.PointCloud()
point_cloud.points = o3d.utility.Vector3dVector(points)
point_cloud.normals = o3d.utility.Vector3dVector(normals)

# Reconstruction de la surface avec la méthode de Poisson
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(point_cloud, depth=10, width=1, scale=1.1)

vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.triangles)

In [ ]:
fig = new_fig()

# Ajouter la surface à la figure
fig.add_trace(go.Mesh3d(
    x=vertices[:, 0],
    y=vertices[:, 1],
    z=vertices[:, 2],
    i=faces[:, 0],
    j=faces[:, 1],
    k=faces[:, 2],
    color='lightgreen'
))

fig.show()

### Point sur la surface

In [ ]:
# Fonction qui indique si un point [x, y, z] est sur la surface mesh ou non avec un precision epsilone
def is_on_surface(mesh, point, eps):
    """
    Indiquer si un point est sur la surface d'un maillage.
    Paramètres :
    - mesh (o3d.geometry.TriangleMesh) : Le maillage de la surface.
    - point (ndarray) : Les coordonnées du point à tester.
    - epsilon (float) : La précision de la vérification.
    Renvoie :
    - is_on_surface (bool) : Vrai si le point est sur la surface, faux sinon.
    """
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.triangles)
    
    for face in faces:
        
        p0 = vertices[face[0]]
        p1 = vertices[face[1]]
        p2 = vertices[face[2]]
        
        normal = np.cross(p1 - p0, p2 - p0)
        normal /= np.linalg.norm(normal)
        
        if np.abs(np.dot(normal, point - p0)) <= eps:
            return True
        
    return False

In [ ]:
# tester is_on_surface
point_on_surface = vertices[0]
print("Le premier point est-il sur la surface ? :", is_on_surface(mesh, point_on_surface, eps=0))

point_not_on_surface = center
print("L'origine est-elle sur la surface ? :", is_on_surface(mesh, [0, 0, 0], eps=1e-3))

point_not_on_surface = center
print("Le centre est-il sur la surface ? :", is_on_surface(mesh, point_not_on_surface, eps=0))

Le premier point est-il sur la surface ? : True
L'origine est-elle sur la surface ? : True
Le centre est-il sur la surface ? : False
